# Nail Health AI - PC-Local Dataset Acquisition and Manifest Generation

This notebook is optimized specifically for local execution on your PC:
1. **Environment Standardization**: Configures paths directly on your local system filesystem.
2. **Credentials Verification**: Verifies your local Kaggle API credential file.
3. **Programmatic Data Acquisition**: Programmatically verifies or downloads the 19 raw fingernail datasets from Kaggle. Skip checks prevent redundant downloading if datasets already exist.
4. **Split & Augmentation Pipeline**: Pools original images, splits them 70/20/10, and augments them within each sealed split to hit balanced target quotas (60,000 total images).
5. **Build Manifest Auditing**: Validates split counts and generates the `dataset_build_manifest.csv` record.

In [ ]:
import os
import sys
from pathlib import Path

# Base directory configuration relative to the workspace
BASE_DIR = Path(os.environ.get('NAILSCAN_BASE_DIR', './modular2_all_in_one')).resolve()
RAW_DIR = BASE_DIR / 'raw'
EXTRACT_DIR = BASE_DIR / 'extracted'
SPLIT_OUTPUT = BASE_DIR / 'final_split_dataset'

print(f"BASE_DIR: {BASE_DIR}")
print(f"RAW_DIR: {RAW_DIR}")
print(f"EXTRACT_DIR: {EXTRACT_DIR}")
print(f"SPLIT_OUTPUT: {SPLIT_OUTPUT}")

In [ ]:
# --- Verify Kaggle API Credentials ---
KAGGLE_CONFIG_DIR = Path.home() / '.kaggle'
kaggle_json_dest = KAGGLE_CONFIG_DIR / 'kaggle.json'

if not kaggle_json_dest.exists():
    local_credential = Path('C:/Users/mjble/CascadeProjects/IMAGETRAINING/kaggle.json')
    if local_credential.exists():
        KAGGLE_CONFIG_DIR.mkdir(parents=True, exist_ok=True)
        import shutil
        shutil.copy2(local_credential, kaggle_json_dest)
        os.chmod(kaggle_json_dest, 0o600)
        print(f"Copied local Kaggle key from: {local_credential}")

if kaggle_json_dest.exists():
    print(f"Kaggle API credential file is active at: {kaggle_json_dest}")
else:
    print(f"WARNING: Kaggle credential file is missing from {kaggle_json_dest}")
    print("Please ensure your 'kaggle.json' file is placed in your home directory's '.kaggle' folder.")

In [ ]:
# --- Dataset Registry & Target Specifications ---
DATASETS = [
    ('arthitaya/nail-dataset', 'nail-dataset'),
    ('sumitagrawal2247134/hp-nail-images-for-early-disease-detection', 'hp-nail-images-for-early-disease-detection'),
    ('naeun1/nail-dataset-disease', 'nail-dataset-disease'),
    ('hunterimam/nail-diseases', 'nail-diseases'),
    ('saisrushikgovindgari/nail-disease-dataset', 'nail-disease-dataset'),
    ('jallipallisaibalaji/nail-and-tongue', 'nail-and-and-tongue'),
    ('saichittamuru/nail-disease-classes', 'nail-disease-classes'),
    ('satyam2527/nail-disease-detection', 'nail-disease-detection'),
    ('jhoncris/nail-image', 'nail-image'),
    ('istiak22/nail-disease', 'nail-disease'),
    ('vedantvaibhav/nail-disease-detection-dataset', 'nail-disease-detection-dataset'),
    ('sdhiman04/nail-disease-dataset', 'sdhiman04_nail-disease-dataset'),
    ('sohomghosal/fingernails-dataset', 'fingernails-dataset'),
    ('hayeon123/finaldata', 'finaldata'),
    ('nikhilgurav21/nail-disease-detection-dataset', 'data'),
    ('josephrasanjana/nail-disease-image-classification-dataset', 'nail_disease_dataset'),
    ('chengcris/nail-disease', 'Dataset'),
    ('mjlebrilla/fingernail-dataset', 'final_dataset_clean'),
    ('mjlebrilla/laeni-dataset-huggingface', 'final_dataset_clean')
]

TARGET_CLASSES = [
    'healthy_nails',
    'koilonychia',
    'pitting',
    'muehrckes_lines',
    'acral_lentiginous_melanoma',
    'clubbing',
    'beau_lines',
    'blue_finger'
]

TRAIN_TARGET_PER_CLASS = 5250
VAL_TARGET_PER_CLASS   = 1500
TEST_TARGET_PER_CLASS  = 750

TARGETS = {
    'train': TRAIN_TARGET_PER_CLASS,
    'val': VAL_TARGET_PER_CLASS,
    'test': TEST_TARGET_PER_CLASS,
}

print(f"Configured {len(DATASETS)} source datasets and {len(TARGET_CLASSES)} target classes.")

In [ ]:
# --- Programmatic Dataset Downloader and Extractor ---
import re
import zipfile
from kaggle.api.kaggle_api_extended import KaggleApi

RAW_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

def _safe_name(slug):
    return re.sub(r'[^A-Za-z0-9._-]+', '_', slug)

os.environ['KAGGLE_CONFIG_DIR'] = str(KAGGLE_CONFIG_DIR)

print("Initializing Kaggle API and authenticating...")
api = KaggleApi()
api.authenticate()
print("Authentication successful.")

for i, (slug, _subfolder) in enumerate(DATASETS, start=1):
    ds_name = _safe_name(slug)
    ds_raw = RAW_DIR / ds_name
    ds_ext = EXTRACT_DIR / ds_name
    ds_raw.mkdir(parents=True, exist_ok=True)
    ds_ext.mkdir(parents=True, exist_ok=True)

    # Skip downloading if images already exist to avoid redundant bandwidth use
    existing_images = any(ds_ext.glob('**/*.*'))
    if existing_images:
        print(f"[{i}/{len(DATASETS)}] Dataset already extracted and ready: {slug}")
        continue

    print(f"[{i}/{len(DATASETS)}] Downloading: {slug}")
    try:
        api.dataset_download_files(slug, path=str(ds_raw), unzip=False, quiet=False)
        
        zips = sorted(ds_raw.glob('*.zip'))
        for z in zips:
            print(f"  Extracting {z.name} -> {ds_ext}")
            with zipfile.ZipFile(z, 'r') as zf:
                zf.extractall(ds_ext)
    except Exception as e:
        print(f"  Error processing {slug}: {e}")

print("Dataset downloading and extraction phase complete.")

In [ ]:
# --- Leakage-Free Split-First, Augment-Second Pipeline ---
import random
import shutil
from PIL import Image, ImageEnhance
import numpy as np

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.gif', '.bmp', '.tiff', '.webp')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def _norm(s):
    return re.sub(r'[^a-z0-9]+', '', s.lower())

ALIAS_TO_TARGET = {}
for tc in TARGET_CLASSES:
    ALIAS_TO_TARGET[_norm(tc)] = tc

manual_mappings = {
    'acral lentiginous melanoma': 'acral_lentiginous_melanoma',
    'acral_lentiginous_melanoma': 'acral_lentiginous_melanoma',
    'beau lines': 'beau_lines',
    'beau_lines': 'beau_lines',
    'blue finger': 'blue_finger',
    'blue_finger': 'blue_finger',
    'clubbing': 'clubbing',
    'healthy nail': 'healthy_nails',
    'healthy_nails': 'healthy_nails',
    'healthynails': 'healthy_nails', 
    'koilonychia': 'koilonychia',
    'koilonychias': 'koilonychia', 
    'muehrckes lines': 'muehrckes_lines',
    'muehrckes_lines': 'muehrckes_lines',
    'pitting': 'pitting'
}
for k, v in manual_mappings.items():
    ALIAS_TO_TARGET[_norm(k)] = v

if SPLIT_OUTPUT.exists():
    print(f"Removing existing split directory at {SPLIT_OUTPUT}")
    shutil.rmtree(SPLIT_OUTPUT)

for folder in ['train', 'val', 'test']:
    for target_class in TARGET_CLASSES:
        (SPLIT_OUTPUT / folder / target_class).mkdir(parents=True, exist_ok=True)

print("STEP 1: Pooling all images per class from datasets...")
class_all_images = {tc: [] for tc in TARGET_CLASSES}
total_processed = 0

for ds_ext_path in EXTRACT_DIR.iterdir():
    if not ds_ext_path.is_dir():
        continue
    for root, _, files in os.walk(ds_ext_path):
        for file in files:
            if Path(file).suffix.lower() in IMAGE_EXTENSIONS:
                current_path = Path(root)
                found_class = None
                parts_to_check = [Path(file).stem] + list(reversed(current_path.parts))
                for part in parts_to_check:
                    normalized_part = _norm(part)
                    if normalized_part in ALIAS_TO_TARGET:
                        found_class = ALIAS_TO_TARGET[normalized_part]
                        break
                if found_class and found_class in TARGET_CLASSES:
                    class_all_images[found_class].append(Path(root) / file)
                total_processed += 1

print(f"Total scanned: {total_processed} | Pooled: {sum(len(v) for v in class_all_images.values())} images.")

print("\nSTEP 2: Splitting originals 70% Train / 20% Val / 10% Test...")
base_train = {}
base_val = {}
base_test = {}

for tc in TARGET_CLASSES:
    imgs = list(class_all_images[tc])
    random.shuffle(imgs)
    n = len(imgs)
    train_end = int(n * 0.70)
    val_end = int(n * 0.90)
    base_train[tc] = imgs[:train_end]
    base_val[tc] = imgs[train_end:val_end]
    base_test[tc] = imgs[val_end:]

def augment_image(img):
    angle = random.uniform(-15, 15)
    img = img.rotate(angle, resample=Image.BICUBIC, expand=False, fillcolor=(0, 0, 0))
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() < 0.2:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
    enhancer = ImageEnhance.Brightness(img)
    img = enhancer.enhance(random.uniform(0.8, 1.2))
    enhancer = ImageEnhance.Contrast(img)
    img = enhancer.enhance(random.uniform(0.8, 1.2))
    if random.random() < 0.3:
        w, h = img.size
        crop_frac = random.uniform(0.05, 0.10)
        left = int(w * crop_frac)
        top = int(h * crop_frac)
        right = w - left
        bottom = h - top
        img = img.crop((left, top, right, bottom)).resize((w, h), Image.BICUBIC)
    return img

augmentation_log = []

def copy_and_augment(image_paths, dest_dir, target_count, split_name, class_name):
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    num_originals = len(image_paths)
    if num_originals == 0:
        return 0
    for i, src in enumerate(image_paths):
        ext = Path(src).suffix
        dest = dest_dir / f"orig_{i:05d}{ext}"
        shutil.copy2(src, dest)
    num_to_augment = target_count - num_originals
    if num_to_augment <= 0:
        if num_originals > target_count:
            all_files = sorted(dest_dir.glob('orig_*'))
            for f in all_files[target_count:]:
                f.unlink()
            return target_count
        return num_originals
    
    aug_count = 0
    attempt = 0
    max_attempts = num_to_augment * 3
    while aug_count < num_to_augment and attempt < max_attempts:
        src_path = random.choice(image_paths)
        try:
            img = Image.open(src_path).convert('RGB')
            aug_img = augment_image(img)
            ext = Path(src_path).suffix
            dest = dest_dir / f"aug_{aug_count:05d}{ext}"
            aug_img.save(dest, quality=95)
            augmentation_log.append({
                'split': split_name,
                'class': class_name,
                'original_path': str(src_path),
                'augmented_path': str(dest)
            })
            aug_count += 1
        except Exception:
            pass
        attempt += 1
    return num_originals + aug_count

print("\nSTEP 3: Augmenting and balancing classes within splits...")
split_configs = [
    ('train', base_train, TRAIN_TARGET_PER_CLASS),
    ('val',   base_val,   VAL_TARGET_PER_CLASS),
    ('test',  base_test,  TEST_TARGET_PER_CLASS),
]

final_counts = {}
for split_name, split_dict, target in split_configs:
    print(f"  Augmenting {split_name.upper()} split...")
    for tc in TARGET_CLASSES:
        dest = SPLIT_OUTPUT / split_name / tc
        count = copy_and_augment(split_dict[tc], dest, target, split_name, tc)
        final_counts[(split_name, tc)] = count

# Save log
augmentation_log_path = BASE_DIR / 'augmentation_mapping_log.csv'
import csv
with augmentation_log_path.open('w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['split', 'class', 'original_path', 'augmented_path']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(augmentation_log)

print(f"\nPipeline Complete. Balanced split dataset saved at: {SPLIT_OUTPUT}")

In [ ]:
# --- Robust Dataset Manifest Audit & Generation ---
import pandas as pd

discovered_classes = set()
for split in ['train', 'val', 'test']:
    split_dir = SPLIT_OUTPUT / split
    if split_dir.exists():
        discovered_classes.update([p.name for p in split_dir.iterdir() if p.is_dir()])

TARGET_CLASSES = sorted(discovered_classes) if discovered_classes else TARGET_CLASSES

build_rows = []
for split in ['train', 'val', 'test']:
    for cls in TARGET_CLASSES:
        class_dir = SPLIT_OUTPUT / split / cls
        if not class_dir.exists():
            orig_count = aug_count = total_count = 0
        else:
            files = [p for p in class_dir.iterdir() if p.is_file()]
            orig_count = sum(p.name.startswith('orig_') for p in files)
            aug_count = sum(p.name.startswith('aug_') for p in files)
            total_count = len(files)
            other_count = total_count - orig_count - aug_count
            if other_count:
                print(f"Note: {other_count} unclassified files in {class_dir}.")

        build_rows.append({
            'class': cls,
            'split': split,
            'original_images': orig_count,
            'augmented_added': total_count - orig_count,
            'final_count': total_count,
            'target': TARGETS.get(split),
        })

build_df = pd.DataFrame(build_rows)
build_csv_path = BASE_DIR / 'dataset_build_manifest.csv'
build_df.to_csv(build_csv_path, index=False)

print('\n=== GENERATED DATASET BUILD MANIFEST ===')
print(build_df.to_string(index=False))
print(f'\nManifest saved to: {build_csv_path}')